In [1]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

import os

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [3]:
def get_block_representations(self, x):
    x = self.stem(x)

    representations = []

    for block in self.blocks:
        x = block(x)
        representations.append(x)

    return representations

In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

In [11]:
class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks=10, channels=128, num_classes=10):
        super().__init__()

        self.channels = channels

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.blocks = nn.ModuleList([
            ResidualBlock(channels)
            for _ in range(num_blocks)
        ])

        self.global_average_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            channels,
            num_classes
        )

    def forward(self, x):
        x = self.stem(x)

        for block in self.blocks:
            x = block(x)

        x = self.global_average_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

    def get_block_representations(self, x):
        x = self.stem(x)

        representations = []

        for block in self.blocks:
            x = block(x)
            representations.append(x)

        return representations

    def get_vectorized_representations(self, x):
        x = self.stem(x)
    
        representations = []
    
        for block in self.blocks:
            x = block(x)
    
            # Global Average Pooling
            vector = x.mean(dim=(2, 3))
    
            representations.append(vector)
    
        return representations

In [13]:
model = ResNetCIFAR(
    num_blocks=10,
    channels=128,
    num_classes=10
)

model = model.to(device)

checkpoint_path = "./resnet_cifar10_epoch1.pth"

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model.eval()

print("Trained model loaded successfully.")

Trained model loaded successfully.


In [15]:
transform = transforms.Compose([
    transforms.ToTensor()
])

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

TEST_SAMPLES_PER_CLASS = 100

test_indices = []

for class_id in range(10):

    class_indices = [
        i for i, label in enumerate(test_dataset.targets)
        if label == class_id
    ]

    test_indices.extend(
        class_indices[:TEST_SAMPLES_PER_CLASS]
    )

small_test_dataset = Subset(
    test_dataset,
    test_indices
)

test_loader = DataLoader(
    small_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Test samples:", len(small_test_dataset))
print("Test batches:", len(test_loader))

Test samples: 1000
Test batches: 8


In [17]:
BATCH_SIZE = 128

test_loader = DataLoader(
    small_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Test batches:", len(test_loader))

Test batches: 8


In [19]:
vectorized_block_features = [
    [] for _ in range(10)
]

all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        representations = (
            model.get_vectorized_representations(images)
        )

        for block_idx, representation in enumerate(
            representations
        ):

            vectorized_block_features[block_idx].append(
                representation.cpu()
            )

        all_labels.append(labels)

print("Vectorized feature extraction completed.")

Vectorized feature extraction completed.


In [21]:
vectorized_block_features = [
    torch.cat(features, dim=0)
    for features in vectorized_block_features
]

all_labels = torch.cat(
    all_labels,
    dim=0
)

print("Number of blocks:", len(vectorized_block_features))
print("Number of labels:", len(all_labels))

Number of blocks: 10
Number of labels: 1000


In [23]:
for block_idx, vectors in enumerate(
    vectorized_block_features
):

    print(
        f"Block {block_idx + 1}:",
        vectors.shape
    )

Block 1: torch.Size([1000, 128])
Block 2: torch.Size([1000, 128])
Block 3: torch.Size([1000, 128])
Block 4: torch.Size([1000, 128])
Block 5: torch.Size([1000, 128])
Block 6: torch.Size([1000, 128])
Block 7: torch.Size([1000, 128])
Block 8: torch.Size([1000, 128])
Block 9: torch.Size([1000, 128])
Block 10: torch.Size([1000, 128])


In [25]:
torch.save(
    vectorized_block_features,
    "./resnet_epoch1_vectorized_representations.pt"
)

torch.save(
    all_labels,
    "./resnet_epoch1_labels.pt"
)

print("Vectorized representations saved successfully.")

Vectorized representations saved successfully.
